# Lab 2.1 Build 1: Chunk and index the corpus

**Track:** cert-od-ara-2-1-core-retrieval-infrastructure  
**Challenge:** Build 1 — Chunk and index the corpus

The 44 Cortex Bank compliance documents are at `/home/elastic/corpus/`. Implement a chunking strategy, index into `cortex-corpus`, and create the alias `cortex-corpus-live`.

In [ ]:
import os, sys, json
sys.path.insert(0, '/opt/ara/lib')
# Load environment
with open('/home/elastic/env') as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ[k] = v
from elasticsearch import Elasticsearch
from ara_index import load_corpus, chunk_fixed, chunk_heading_aware, chunk_unit_preserving, Chunk, index_chunks, chunk_report
es = Elasticsearch(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'], request_timeout=60)
print('Setup complete. ES connected:', es.info()['version']['number'])

## Step 1: Load the corpus

Load all 44 documents from `/home/elastic/corpus/`.

In [ ]:
docs = load_corpus('/home/elastic/corpus')
print(f'Loaded {len(docs)} documents')
print(f'Example: {docs[0]["doc_id"]} ({docs[0]["doc_type"]})')

## Step 2: Implement your chunking strategy

Implement the `chunk` function below. The function receives one document dict and returns a list of `Chunk` objects.

Available strategies: `chunk_fixed(doc, max_tokens, overlap_tokens)`, `chunk_heading_aware(doc, max_tokens)`, `chunk_unit_preserving(doc, units, max_tokens)`.

Constraints:
- Median chunk must be at most 512 tokens
- Designated semantic units (risk tables, deadline schedules) must survive intact

In [ ]:
# ── YOUR WORK ──
# Example: chunk_heading_aware(doc, max_tokens=512)
def chunk(doc):
    """Return a list of Chunk objects from one document."""
    return chunk_heading_aware(doc, max_tokens=512)
# ──────────────

In [ ]:
all_chunks = []
for doc in docs:
    all_chunks.extend(chunk(doc))
report = chunk_report(all_chunks)
print('Chunk report:', json.dumps(report, indent=2))
if report['median_tokens'] <= 512:
    print('Median OK (<= 512 tokens)')
else:
    print(f'Median too high ({report["median_tokens"]} > 512) -- adjust your strategy')

## Step 3: Write the mapping in Kibana Dev Tools

Open the **Kibana** tab and run:

```json
DELETE /cortex-corpus

PUT /cortex-corpus
{
  "mappings": {
    "properties": {
      "body":           { "type": "semantic_text", "inference_id": "<ARA_EMBED_ID from env>" },
      "body_text":      { "type": "text" },
      "doc_id":         { "type": "keyword" },
      "doc_type":       { "type": "keyword" },
      "title":          { "type": "keyword" },
      "section":        { "type": "keyword" },
      "effective_date": { "type": "date" }
    }
  }
}
```

Then come back here to index.

In [ ]:
# ── YOUR WORK ──
# Run this cell after creating the mapping in Kibana Dev Tools
result = index_chunks(es, 'cortex-corpus', all_chunks)
print('Index result:', result)
# ──────────────

## Step 4: Create the alias

In Kibana Dev Tools:

```json
POST /_aliases
{
  "actions": [
    { "add": { "index": "cortex-corpus", "alias": "cortex-corpus-live" } }
  ]
}
```

In [ ]:
# Verify alias
try:
    aliases = es.indices.get_alias(name='cortex-corpus-live')
    print('cortex-corpus-live ->', list(aliases.keys()))
except Exception as e:
    print(f'Alias not found yet: {e}')

## Step 5: Run the dev evaluation

From the **Terminal** tab:

```bash
source /home/elastic/env
cd /home/elastic/dev-sets
python3 eval-chunking.py
```

When median is <= 512 and cortex-corpus-live resolves, select **Check**.